<a href="https://colab.research.google.com/github/sauravv19/A-Security-Analysis-of-Tf-Idf-And-Deep-Learning-Model-for-Hate-Speech-Detection-in-YouTube/blob/main/Hate%20Speech%20Detection%20in%20YouTube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask flask-cors pyngrok yt-dlp -q

In [ ]:
import torch
import numpy as np
from scipy.special import softmax
from transformers import (
    AutoTokenizer,
    RobertaForSequenceClassification,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    pipeline
)
from subprocess import run, PIPE
from uuid import uuid4
from os import makedirs, remove

In [ ]:
MAX_SEQUENCE_LENGTH = 512
OVERLAP             = 50
STRIDE              = MAX_SEQUENCE_LENGTH - OVERLAP
MAX_NEW_TOKENS      = 128
CHUNK_LENGTH        = 30

In [ ]:
class ASRHate:

    def _build_pipe(self):
        self.asr_pipe = pipeline(
            "automatic-speech-recognition",
            model=self.wh_model,
            tokenizer=self.wh_processor.tokenizer,
            feature_extractor=self.wh_processor.feature_extractor,
            chunk_length_s=CHUNK_LENGTH,
            device=self.device,
            generate_kwargs={"max_new_tokens": MAX_NEW_TOKENS},
        )
        return self

    def _to_device(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.wh_model.to(self.device)
        return self

    @staticmethod
    def _download(video_url: str) -> str:
        makedirs("/content/outputs", exist_ok=True)
        out = f"/content/outputs/output-{uuid4()}.wav"
        proc = run(
            f'yt-dlp --extract-audio --audio-format wav --output {out} {video_url}',
            shell=True, stderr=PIPE, stdout=PIPE
        )
        if proc.returncode == 0:
            return out
        raise Exception(proc.stderr.decode(errors="ignore"))

    def transcribe(self, video_url: str):
        self.audio_path   = ASRHate._download(video_url)
        self.transcription = self.asr_pipe(self.audio_path)
        return self

    def classify(self):
        inputs = self.classifier_tokenizer(
            self.transcription["text"],
            max_length=MAX_SEQUENCE_LENGTH,
            padding="max_length",
            truncation=True,
            return_overflowing_tokens=True,
            stride=STRIDE,
            return_tensors="pt"
        )
        model_inputs = {
            "input_ids":      inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
        }
        with torch.no_grad():
            outputs = self.classifier_model(**model_inputs)
        logits_np = outputs.logits.cpu().numpy() if self.device == 'cuda' else outputs.logits.numpy()
        self.probabilities = softmax(logits_np, axis=1)
        return self

    def clean(self):
        remove(self.audio_path)
        print("Removed:", self.audio_path)
        del self.audio_path

    def chunkwise(self):
        percentages = (self.probabilities * 100).tolist()
        label_0 = self.classifier_model.config.id2label[0].title()
        label_1 = self.classifier_model.config.id2label[1].title()
        for i, pcts in enumerate(percentages):
            print(f"Chunk {i}: {label_0} {pcts[0]:.2f}%  {label_1} {pcts[1]:.2f}%")

    def overall(self):
        mean_pcts = (np.mean(self.probabilities, axis=0) * 100).tolist()
        for j, pct in enumerate(mean_pcts):
            label = self.classifier_model.config.id2label[j]
            print(f"Mean {label.title()}: {pct:.2f}%")

    def __init__(self):
        print("Loading Whisper...")
        self.wh_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")
        self.wh_model     = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo")
        print("Loading RoBERTa...")
        self.classifier_tokenizer = AutoTokenizer.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
        self.classifier_model     = RobertaForSequenceClassification.from_pretrained("facebook/roberta-hate-speech-dynabench-r4-target")
        self._to_device()._build_pipe()
        print(" All models loaded! Device:", self.device)

asr_hate = ASRHate()

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

In [ ]:
ngrok.set_auth_token("3DGEtLbPZ5IfJ1ZP8NYbxjRG9Gg_55ZQJFP1oPKYYgErVcztj")

# ── Detect which column = hate vs nothate ─────────────────────────
id2label = asr_hate.classifier_model.config.id2label
print("Model labels:", id2label)
# facebook/roberta-hate-speech-dynabench-r4-target → {0: 'nothate', 1: 'hate'}

hate_col    = next(k for k, v in id2label.items() if 'hate' in v.lower() and 'not' not in v.lower())
nothate_col = next(k for k, v in id2label.items() if 'not' in v.lower())
print(f"hate_col={hate_col}, nothate_col={nothate_col}")

# ── Flask app ─────────────────────────────────────────────────────
app = Flask(__name__)
CORS(app)

@app.after_request
def skip_ngrok_warning(response):
    response.headers['ngrok-skip-browser-warning'] = 'true'
    return response

@app.route('/analyze', methods=['POST'])
def analyze():
    data   = request.json
    yt_url = data.get('url', '').strip()
    if not yt_url:
        return jsonify({'error': 'No URL provided'}), 400

    try:
        print(f"\n→ Analyzing: {yt_url}")

        asr_hate.transcribe(yt_url)
        print(" Transcribed:", asr_hate.transcription['text'][:100])

        asr_hate.classify()
        n_chunks = asr_hate.probabilities.shape[0]
        print(f" Classified {n_chunks} chunks")

        asr_hate.clean()

        probs       = asr_hate.probabilities      # (N, 2)
        percentages = probs * 100

        chunks = []
        for i, row in enumerate(percentages.tolist()):
            h = row[hate_col]
            n = row[nothate_col]
            chunks.append({
                'index':       i,
                'hate_pct':    round(h, 2),
                'nothate_pct': round(n, 2),
                'label':       'hate' if h >= n else 'nothate',
            })

        mean_probs       = np.mean(probs, axis=0)
        mean_hate_pct    = float(mean_probs[hate_col]    * 100)
        mean_nothate_pct = float(mean_probs[nothate_col] * 100)
        hate_count       = sum(1 for c in chunks if c['label'] == 'hate')
        nothate_count    = len(chunks) - hate_count

        print(f" Result — {hate_count} hate / {nothate_count} non-hate chunks")
        print(f"   Overall: hate={mean_hate_pct:.1f}%  nothate={mean_nothate_pct:.1f}%")

        return jsonify({
            'transcription': asr_hate.transcription['text'],
            'chunks':        chunks,
            'overall': {
                'hate_pct':    round(mean_hate_pct,    2),
                'nothate_pct': round(mean_nothate_pct, 2),
                'verdict':     'hate' if mean_hate_pct >= mean_nothate_pct else 'nothate',
            },
            'total_chunks':  len(chunks),
            'hate_count':    hate_count,
            'nothate_count': nothate_count,
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

# ── Launch ────────────────────────────────────────────────────────
public_url = ngrok.connect(5000)
print("\n" + "="*60)
print("    Backend is LIVE!")
print(f"    Paste this URL into the frontend:")
print(f"      {public_url}")
print("="*60 + "\n")

app.run(port=5000, use_reloader=False)